# 4.13 Cross Validation Basics

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 4: Data Preprocessing & Exploratory Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-04-data-preprocessing-and-exploratory-data-analysis/04.13_cross-validation-basics.ipynb)

Companion code for Section 4.13 of the book. Run the cells in order from top to bottom.

**Example 4.13.1**

In [ ]:
import numpy as np
from sklearn.model_selection import (
    KFold, StratifiedKFold, cross_val_score, cross_validate,
    LeaveOneOut, RepeatedStratifiedKFold, TimeSeriesSplit
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X, y = load_breast_cancer(return_X_y=True)

# ── Standard k-fold ───────────────────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = LogisticRegression(max_iter=1000)

scores_kf = cross_val_score(model, X, y, cv=kf, scoring='accuracy')
print(f'5-Fold CV: {scores_kf.round(4)}')
print(f'Mean ± Std: {scores_kf.mean():.4f} ± {scores_kf.std():.4f}')

# ── Stratified k-fold - preserves class balance in each fold ──────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_skf = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
print(f'\nStratified 5-Fold: {scores_skf.round(4)}')
print(f'Mean ± Std: {scores_skf.mean():.4f} ± {scores_skf.std():.4f}')

# ── Multiple metrics at once ───────────────────────────────────────
# Use Pipeline to prevent data leakage - scaler fitted only on train fold
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=1000))
])

cv_results = cross_validate(
    pipeline, X, y, cv=skf,
    scoring=['accuracy','precision_macro','recall_macro','f1_macro','roc_auc'],
    return_train_score=True
)

print('\nCross-validation results (5-fold Stratified):')
for metric, values in cv_results.items():
    if 'test_' in metric:
        name = metric.replace('test_', '')
        print(f'  {name:20s}: {values.mean():.4f} ± {values.std():.4f}')

# Check for overfitting: train score >> test score
print(f'\nOverfit check:')
print(f'  Train accuracy: {cv_results["train_accuracy"].mean():.4f}')
print(f'  Test accuracy:  {cv_results["test_accuracy"].mean():.4f}')
gap = cv_results['train_accuracy'].mean() - cv_results['test_accuracy'].mean()
print(f'  Gap: {gap:.4f} ({"OK" if gap < 0.05 else "OVERFITTING"})')

# ── Choosing k ────────────────────────────────────────────────────
print('\nEffect of k on CV estimate:')
for k in [3, 5, 10, 20]:
    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    s  = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
    print(f'  k={k:2d}: mean={s.mean():.4f}  std={s.std():.4f}  (higher k → lower variance, slower)')

**Example 4.13.2**

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, RepeatedStratifiedKFold
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

X, y = load_breast_cancer(return_X_y=True)

# ── Repeated Stratified K-Fold ────────────────────────────────────
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
scores_rep = cross_val_score(model, X, y, cv=rskf, scoring='roc_auc')
print(f'Repeated (5×10) StratifiedKFold AUC:')
print(f'  50 evaluations: mean={scores_rep.mean():.4f} ± {scores_rep.std():.4f}')

# ── Time-series CV ────────────────────────────────────────────────
tscv = TimeSeriesSplit(n_splits=5, gap=0)  # gap: skip samples between train/test
print(f'\nTime Series CV split sizes:')
for i, (train, test) in enumerate(tscv.split(X)):
    print(f'  Fold {i+1}: train={len(train):3d}, test={len(test):3d}  '
          f'(train [{train[0]}:{train[-1]}], test [{test[0]}:{test[-1]}])')

# ── Nested CV - model selection without test set bias ─────────────
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv  = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

param_grid = {'n_estimators': [50, 100], 'max_depth': [None, 5]},
clf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid[0], cv=inner_cv)

nested_scores = cross_val_score(clf, X, y, cv=outer_cv, scoring='roc_auc')
print(f'\nNested CV (outer=5, inner=3) AUC:')
print(f'  Scores: {nested_scores.round(4)}')
print(f'  Mean:   {nested_scores.mean():.4f} ± {nested_scores.std():.4f}')
print(f'  (This is an unbiased estimate even with hyperparameter tuning)')